# Prototyp: Recherche-Pipeline mit BM25 und Vektorsuche

**Hackathon Algorithmen & Datenstrukturen, Gruppe 2 – Phase 3: Erste Implementierung**

**Problem:** Bei der Recherche mit KI kommt es häufig zu Halluzinationen, und das Sprachmodell verbraucht viele Tokens, weil es selbst sucht und liest.

**Idee:** Ein regelbasierter Algorithmus übernimmt die Recherche. Er holt Fachartikel von arXiv, zerlegt sie in Abschnitte (Chunks), wählt per BM25 und Vektorsuche die passendsten aus und gibt nur diese mit Quelle zurück. Später läuft das als MCP-Tool, das ein Sprachmodell aufrufen kann.

**Ablauf** (siehe `Phase2/Flowchart_RAG-Pipeline.drawio`):

1. Schlagwörter prüfen und normalisieren
2. Dokumente von arXiv holen (Titel und Abstract)
3. In überlappende Chunks zerlegen
4. Keyword-Vorauswahl per BM25: die besten m Chunks
5. Mit Einbetter: Vektorsuche (Voyage AI), Rangfusion (RRF) und Duplikat-Filter (MMR); ohne Einbetter: die besten k nach BM25
6. Ausgabe als JSON mit Quelle

Das Notebook läuft ohne Anpassung von oben nach unten durch. Ohne `VOYAGE_API_KEY` arbeitet es nur mit BM25, ohne Internet mit eingebauten Beispieldokumenten.

## 0. Einrichtung

In [1]:
import sys
from pathlib import Path

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "pipeline").is_dir())
sys.path.insert(0, str(ROOT / "src"))
print("Projektordner:", ROOT)
print("Python:", sys.version.split()[0])

Projektordner: /Users/lucafeest/PycharmProjects/AlgorithmusProjekt
Python: 3.15.0


In [2]:
import json
import time
from datetime import date

import bm25
import chunking
import ranking
import vector_search
from pipeline import PipelineConfig, research
from sources import ArxivSource, Document, Source, SourceError

## 1. Die Bausteine einzeln

Jeder Baustein ist eine eigene Funktion mit Pseudocode in `docs/<modul>/` und Unittests in `src/<modul>/`.

### 1.1 Tokenizer (BM25)

Kleinschreibung mit `casefold`, Bindestrich und Apostroph verbinden, alle anderen Satzzeichen trennen.

In [3]:
bm25.tokenize("Der Apfel-Baum ist grün. E-Mail, COVID-19, Straße – don't!")

['der', 'apfelbaum', 'ist', 'grün', 'email', 'covid19', 'strasse', 'dont']

### 1.2 Invertierter Index und BM25

Mini-Korpus aus der Formelsammlung (`docs/bm25/bm25-formelsammlung.md`). Die Handrechnung für „apfel milch“ ergibt Dok 3 = 0,767 und Dok 0 = 0,694. Dok 2 besteht nur aus Satzzeichen und wird übersprungen.

In [4]:
korpus = ["Apfel apfel Banane.", "Banane, Milch!", ".!?", "Apfel Brot Milch Käse Butter"]
index, laengen, n_docs, avgdl = bm25.build_index(korpus)
print("Index:", index)
print("Längen:", laengen, "| N =", n_docs, "| avgdl =", round(avgdl, 3))
for doc_id, score in bm25.top_m("apfel milch", index, laengen, n_docs, avgdl, 10):
    print(f"Dok {doc_id}: {score:.3f}  ({korpus[doc_id]})")

Index: {'apfel': {0: 2, 3: 1}, 'banane': {0: 1, 1: 1}, 'milch': {1: 1, 3: 1}, 'brot': {3: 1}, 'käse': {3: 1}, 'butter': {3: 1}}
Längen: {0: 3, 1: 2, 3: 5} | N = 3 | avgdl = 3.333
Dok 3: 0.767  (Apfel Brot Milch Käse Butter)
Dok 0: 0.694  (Apfel apfel Banane.)
Dok 1: 0.573  (Banane, Milch!)


Mit `k1` steuert man, wie viel Wiederholung zählt, mit `b`, wie stark lange Dokumente bestraft werden. Dok 0 („apfel apfel“) überholt Dok 3 („apfel … milch“) erst bei großem `k1` und nur, wenn `b > 0`.

In [5]:
print(" k1     b |  Dok 0  Dok 3 | vorne")
for k1 in (0.5, 1.5, 3, 10):
    for b in (0, 0.75, 1):
        scores = dict(bm25.top_m("apfel milch", index, laengen, n_docs, avgdl, 10, k1=k1, b=b))
        vorne = 0 if scores[0] > scores[3] else 3
        print(f"{k1:>3} {b:>5} | {scores[0]:6.3f} {scores[3]:6.3f} | Dok {vorne}")

 k1     b |  Dok 0  Dok 3 | vorne
0.5     0 |  0.564  0.940 | Dok 3
0.5  0.75 |  0.573  0.836 | Dok 3
0.5     1 |  0.576  0.806 | Dok 3
1.5     0 |  0.671  0.940 | Dok 3
1.5  0.75 |  0.694  0.767 | Dok 3
1.5     1 |  0.701  0.723 | Dok 3
  3     0 |  0.752  0.940 | Dok 3
  3  0.75 |  0.787  0.734 | Dok 0
  3     1 |  0.800  0.684 | Dok 0
 10     0 |  0.862  0.940 | Dok 3
 10  0.75 |  0.919  0.701 | Dok 0
 10     1 |  0.940  0.646 | Dok 0


### 1.3 Chunking

Wörter statt LLM-Tokens; benachbarte Chunks teilen `overlap` Wörter, damit ein Satz an der Grenze nicht verloren geht.

In [6]:
chunking.split_into_chunks("w1 w2 w3 w4 w5 w6 w7 w8 w9 w10", chunk_size=4, overlap=1)

['w1 w2 w3 w4', 'w4 w5 w6 w7', 'w7 w8 w9 w10']

### 1.4 Vektorsuche, Rangfusion und Duplikat-Filter

Rechenbeispiele aus `docs/vector_search/vector_search-formelsammlung.md` und `docs/ranking/ranking-formelsammlung.md`.

In [7]:
print("Cosinus (1,2,2) und (2,0,1):", round(vector_search.cosine_similarity([1, 2, 2], [2, 0, 1]), 3))
print("Top-3 per Min-Heap:", vector_search.top_k([0.42, 0.17, 0.93, 0.05, 0.61], 3))
print("RRF, Listen [1, 2] und [2, 3], k_rrf = 0:", ranking.reciprocal_rank_fusion([[1, 2], [2, 3]], k_rrf=0))

vektoren = [[1.0, 0.0], [1.0, 0.01], [0.0, 1.0]]
relevanz = [1.0, 0.95, 0.6]
print("MMR mit λ = 1 (nur Relevanz):", vector_search.mmr_select(relevanz, vektoren, 2, 1.0))
print("MMR mit λ = 0,5 (fast-Duplikat 1 fliegt raus):", vector_search.mmr_select(relevanz, vektoren, 2, 0.5))

Cosinus (1,2,2) und (2,0,1): 0.596
Top-3 per Min-Heap: [(2, 0.93), (4, 0.61), (0, 0.42)]
RRF, Listen [1, 2] und [2, 3], k_rrf = 0: [(2, 1.5), (1, 1.0), (3, 0.5)]
MMR mit λ = 1 (nur Relevanz): [0, 1]
MMR mit λ = 0,5 (fast-Duplikat 1 fliegt raus): [0, 2]


## 2. Die ganze Pipeline mit echten Daten

Die Dokumente werden einmal von arXiv geholt und dann für alle Läufe wiederverwendet. So sind die Vergleiche fair, und arXiv wird nicht mehrfach abgefragt. Ohne Internet springen drei eingebaute Beispieldokumente ein: zwei echte arXiv-Einträge mit gekürztem Abstract und ein erfundenes Gegenbeispiel.

In [8]:
SCHLAGWOERTER = ["UML", "class diagram", "semantics"]
FRAGE = "Wie definiert man die Semantik von UML-Klassendiagrammen formal?"
K = 5

BEISPIELE = [
    Document("1409.6635", "System Model Semantics of Class Diagrams",
             "Defining semantics for UML is a difficult task. We use a system model to give class diagrams a precise semantics.",
             ("Maria Victoria Cengarle", "Hans Grönninger", "Bernhard Rumpe"), date(2014, 9, 22),
             "https://arxiv.org/abs/1409.6635", "beispiel"),
    Document("2210.14441", "Towards Automatically Extracting UML Class Diagrams from Natural Language Specifications",
             "We propose an automated approach for the extraction of UML class diagrams from natural language software specifications.",
             ("Song Yang", "Houari Sahraoui"), date(2022, 10, 26), "https://arxiv.org/abs/2210.14441", "beispiel"),
    Document("0000.00001", "Neural networks for image recognition",
             "Convolutional networks recognise objects in images.",
             ("Jane Doe",), date(2020, 1, 1), "https://example.org/1", "beispiel"),
]


class GespeicherteQuelle(Source):
    name = "gespeichert"

    def __init__(self, dokumente):
        self.dokumente = dokumente

    def search(self, keywords, max_results):
        return self.dokumente[:max_results]


try:
    dokumente = ArxivSource(categories=("cs.SE",)).search(SCHLAGWOERTER, 20)
    print(f"{len(dokumente)} Dokumente von arXiv geholt.")
except SourceError as fehler:
    dokumente = BEISPIELE
    print("arXiv nicht erreichbar, nutze Beispieldokumente:", fehler)

QUELLE = GespeicherteQuelle(dokumente)


def zeige(ergebnis):
    for platz, treffer in enumerate(ergebnis.hits, start=1):
        print(f"{platz}. {treffer.score:7.4f}  {treffer.document.id:12}  {treffer.document.title[:70]}")
    for hinweis in ergebnis.warnings:
        print("Hinweis:", hinweis)

20 Dokumente von arXiv geholt.


### 2.1 Nur BM25 (ohne Einbetter)

In [9]:
start = time.perf_counter()
nur_bm25 = research(SCHLAGWOERTER, K, QUELLE)
print(f"{(time.perf_counter() - start) * 1000:.0f} ms")
zeige(nur_bm25)

3 ms
1.  3.0081  1409.6635     System Model Semantics of Class Diagrams
2.  2.9236  1409.6928     Developing the UML as a Formal Modelling Notation
3.  2.6141  2011.07313    Classification of Reverse-Engineered Class Diagram and Forward-Enginee
4.  2.5998  1409.6919     The UML as a Formal Modeling Notation
5.  2.5199  1211.0713     From user requirements to UML class diagram


### 2.2 Hybrid mit Voyage AI

Die Anfrage wird als „query“ eingebettet (hier die deutsche Frage), die Kandidaten aus der BM25-Vorauswahl als „document“. Ohne `VOYAGE_API_KEY` oder bei einem Fehler der API liefert die Pipeline BM25-Ergebnisse mit Hinweis.

In [10]:
try:
    EINBETTER = vector_search.VoyageEmbedder()
except vector_search.EmbeddingError:
    EINBETTER = None
    print("Kein VOYAGE_API_KEY gesetzt, deshalb nur BM25.")

start = time.perf_counter()
hybrid = research(SCHLAGWOERTER, K, QUELLE, EINBETTER, question=FRAGE)
print(f"{(time.perf_counter() - start) * 1000:.0f} ms")
zeige(hybrid)

2194 ms
1.  0.0328  1409.6635     System Model Semantics of Class Diagrams
2.  0.0323  1409.6928     Developing the UML as a Formal Modelling Notation
3.  0.0315  1409.6919     The UML as a Formal Modeling Notation
4.  0.0306  1606.02311    A Formal Semantic for UML 2.0 Activity Diagram based on Institution Th
5.  0.0294  2011.07313    Classification of Reverse-Engineered Class Diagram and Forward-Enginee


### 2.3 Vergleich

In [11]:
print(f"{'Platz':<6}{'nur BM25':<50}{'hybrid':<50}")
for platz in range(K):
    links = nur_bm25.hits[platz].document.title[:46] if platz < len(nur_bm25.hits) else ""
    rechts = hybrid.hits[platz].document.title[:46] if platz < len(hybrid.hits) else ""
    print(f"{platz + 1:<6}{links:<50}{rechts:<50}")

Platz nur BM25                                          hybrid                                            
1     System Model Semantics of Class Diagrams          System Model Semantics of Class Diagrams          
2     Developing the UML as a Formal Modelling Notat    Developing the UML as a Formal Modelling Notat    
3     Classification of Reverse-Engineered Class Dia    The UML as a Formal Modeling Notation             
4     The UML as a Formal Modeling Notation             A Formal Semantic for UML 2.0 Activity Diagram    
5     From user requirements to UML class diagram       Classification of Reverse-Engineered Class Dia    


## 3. Ausgabe für das Sprachmodell

So sieht die Antwort aus, die das MCP-Tool zurückgibt: nur die ausgewählten Chunks, jeweils mit Quelle.

In [12]:
print(json.dumps(hybrid.to_dict()["hits"][0], ensure_ascii=False, indent=2))

{
  "document_id": "1409.6635",
  "title": "System Model Semantics of Class Diagrams",
  "authors": [
    "Maria Victoria Cengarle",
    "Hans Grönninger",
    "Bernhard Rumpe"
  ],
  "published": "2014-09-22",
  "url": "https://arxiv.org/abs/1409.6635v1",
  "source": "arxiv",
  "chunk_position": 0,
  "text": "System Model Semantics of Class Diagrams Defining semantics for UML is a difficult task. Disagreements in the meaning of UML constructs as well as the size of UML are major obstacles. In this report, we describe our approach to define the semantics for UML. Semantics is defined denotationally as a mapping into our semantics domain called the system model [4, 5, 6]. We demonstrate our approach by defining the semantics for a comprehensive version of class diagrams. The semantics definition is detailed for UML/P class diagrams, a variant of class diagrams which restricts the use of a few methodologically and semantically involved concepts. Class diagrams are well-known and rather e

**Grobe Schätzung der Ersparnis** (etwa 4 Zeichen pro Token): Ohne Tool müsste das Modell alle gefundenen Abstracts lesen, mit Tool nur die Auswahl. Die echte Messung mit Tokenzählung der API folgt in Phase 4.

In [13]:
zeichen_alle = sum(len(dokument.text) for dokument in dokumente)
zeichen_auswahl = len(json.dumps(hybrid.to_dict(), ensure_ascii=False))
print(f"Alle {len(dokumente)} Abstracts: {zeichen_alle:>6} Zeichen ≈ {zeichen_alle // 4:>5} Tokens")
print(f"Auswahl ({len(hybrid.hits)} Chunks als JSON): {zeichen_auswahl:>6} Zeichen ≈ {zeichen_auswahl // 4:>5} Tokens")
print(f"Ersparnis: {1 - zeichen_auswahl / zeichen_alle:.0%}")

Alle 20 Abstracts:  22642 Zeichen ≈  5660 Tokens
Auswahl (5 Chunks als JSON):   6991 Zeichen ≈  1747 Tokens
Ersparnis: 69%


## 4. Fehler- und Sonderfälle

Pflicht laut Hackathon-Regel 4. Jeder Fall ist zusätzlich als Unittest abgesichert.

In [14]:
class AusgefalleneQuelle(Source):
    name = "ausgefallen"

    def search(self, keywords, max_results):
        raise SourceError("Verbindung abgelehnt")


class AusgefallenerEinbetter(vector_search.Embedder):
    def _request(self, texts, input_type):
        raise vector_search.EmbeddingError("HTTP 429")


faelle = [
    ("Leere Schlagwörter", lambda: research(["  ", ""], K, QUELLE)),
    ("k = 0", lambda: research(SCHLAGWOERTER, 0, QUELLE)),
    ("Quelle nicht erreichbar", lambda: research(SCHLAGWOERTER, K, AusgefalleneQuelle())),
    ("Quelle findet nichts", lambda: research(SCHLAGWOERTER, K, GespeicherteQuelle([]))),
    ("Kein Chunk enthält ein Schlagwort", lambda: research(["quantenchromodynamik"], K, QUELLE)),
    ("Einbetter fällt aus", lambda: research(SCHLAGWOERTER, K, QUELLE, AusgefallenerEinbetter())),
]
for name, aufruf in faelle:
    try:
        ergebnis = aufruf()
        print(f"{name:<36} → {len(ergebnis.hits)} Treffer, Hinweise: {list(ergebnis.warnings)}")
    except (ValueError, SourceError) as fehler:
        print(f"{name:<36} → {type(fehler).__name__}: {fehler}")

Leere Schlagwörter                   → ValueError: keywords must not be empty
k = 0                                → ValueError: k must be > 0
Quelle nicht erreichbar              → SourceError: Verbindung abgelehnt
Quelle findet nichts                 → 0 Treffer, Hinweise: ['Keine Treffer']
Kein Chunk enthält ein Schlagwort    → 0 Treffer, Hinweise: ['Kein Chunk enthält ein Schlagwort']
Einbetter fällt aus                  → 5 Treffer, Hinweise: ['Vektorsuche nicht verfügbar, nur BM25']


## 5. Bekannte Grenzen (Auswahl)

- arXiv ist englischsprachig; deutsche Schlagwörter finden kaum etwas. Die Vektorsuche gleicht das bei der **Frage** aus, nicht bei der Suche selbst.
- Nur Titel und Abstract, kein Volltext.
- Chunks werden in Wörtern gemessen, nicht in LLM-Tokens.
- Fast-Duplikate (z. B. zwei Fassungen desselben Papers) rutschen mit λ = 0,7 durch MMR.
- Die Ersparnis oben ist eine Schätzung; gemessen wird in Phase 4.
- Ohne Voyage-Schlüssel fehlt die Vektorsuche; das Ergebnis ist dann reines BM25.

Ausführlicher in den Pseudocode-Dateien unter „Bekannte Grenzen“ und in der Reflexion (Phase 5).